## Set Up

In [1]:
# Import Libraries

import json                 # Reading and writing JSON
from glob import glob       # File pattern matching
import os                   # Operating system access
import pathlib              # File navigation
import holoviews as hv      # Interactable plot
import panel as pn          # Plot formatting
pn.extension()              # Plot formatting
hv.extension('bokeh')       # Interactable plot
import numpy as np

import earthpy              # Work with geospatial data
import xarray as xr         # Multi-dimensional arrays
import hvplot.xarray        # HVPlot and xarray
import rioxarray as rxr     # Raster for xarray

import matplotlib.pyplot as plt    # Plotting
import pandas as pd         # Work with tabular data
import geopandas as gpd     # Wowk with geospatial data
import hvplot.pandas        # Plot geospatial data

In [2]:
project = earthpy.Project(
    'Gila River Vegetation', dirname='NDVI_data')
project.get_data()

## Establish Boundaries

In [3]:
# Load in the boundary data
aitsn_gdf = gpd.read_file(project.project_dir / 'tl_2020_us_aitsn')

# Check that it worked
print(type(aitsn_gdf))

aitsn_gdf.head()

In [4]:
overview_plot = aitsn_gdf.hvplot(
    geo=True, tiles='CartoLight',
    frame_width=500,
    legend=False, fill_color=None, fill_alpha = .5,
    line_color='blue',
    colorbar = False,
    hover_cols='all',
    xlim=(-112.4, -111.4),
    ylim=(32.9, 33.6),
    title = 'GRIC & SRIC Boundary Overview'
)

In [ ]:
overview_plot

> Need to figure out how to save the above plot as html

In [5]:
gric_gdf = aitsn_gdf.loc[aitsn_gdf.AIANNHCE=='1310'].dissolve()

gric_gdf.hvplot(
    geo=True, tiles='CartoLight',
    fill_color='green', line_color='Blue',
    fill_alpha=.3,  # must be b/w 0-1
    title='Gila River Indian Community',
    frame_width=500)

In [6]:
sric_gdf = aitsn_gdf.loc[aitsn_gdf.AIANNHCE=='3340'].dissolve()

sric_gdf.hvplot(
    geo=True, tiles='CartoLight',
    fill_color='green', line_color='Blue',
    fill_alpha=.3,  # must be b/w 0-1
    title='Salt River Indian Community',
    frame_width=500)

## Download Data
> The GRIC NDVI data is included in the download for the AITSN Boundary data used above

In [ ]:
import earthpy.api.appeears as eaapp

In [ ]:
ndvi_downloader = eaapp.AppeearsDownloader(

    ### give your download a name
    download_key = "salt-river-ndvi",

    ### tell it to put the data in your project that you already defined
    project = project,

    ### specify the MODIS product you want
    product = 'MOD13Q1.061',
    layer = '_250m_16_days_NDVI',

    ### choose a start date and end data
    start_date = "05-24",
    end_date = "08-29",

    ### recurring means you want those dates over multiple years
    recurring = True,

    ### specify the range of years you want
    year_range = [2001, 2022],

    ### specify the polygon you want to get NDVI data for
    polygon = sric_gdf
)

In [ ]:
ndvi_downloader.download_files(cache=True)

> APPEARS API is currently down due to technical difficulties

In [7]:
# Get a sorted list of file path for only the 'gila_river-ndvi' folder
gila_dir = project.project_dir / 'gila-river-ndvi'

gric_paths = sorted(list(gila_dir.rglob('*NDVI*.tif')))

# Display the first and last three files paths to check the pattern
gric_paths[:3], gric_paths[-3:]

In [8]:
doy_start = -25
doy_end = -19

gric_das = []

for gric_path in gric_paths:
    # Get date from file name
    doy = gric_path.name[doy_start:doy_end]
    date = pd.to_datetime(doy, format='%Y%j')

    # Open dataset
    da = rxr.open_rasterio(gric_path, mask_and_scale=True).squeeze()

    # Add date dimension and clean up metadata
    da = da.assign_coords({'date': date})
    da = da.expand_dims({'date': 1})
    da.name = 'NDVI'

    # Prepare for concatenation
    gric_das.append(da)

len(gric_das)

In [9]:
# Combine NDVI images from all dates
gric_da = xr.combine_by_coords(gric_das, coords=['date'])

gric_da

In [10]:
# Compute the mean NDVI values before water rights were restored
gric_ndvi_pre = (gric_da
             .sel(date=slice('2001', '2004'))
             .mean('date')
             .NDVI
)

gric_ndvi_pre.head()

In [11]:
gric_ndvi_post = gric_da.sel(date=slice('2005', '2022')).NDVI

gric_ndvi_post_yearly = gric_ndvi_post.groupby('date.year').mean('date')

print (gric_ndvi_post_yearly)

len(gric_ndvi_post_yearly)

In [12]:
gric_diff = gric_ndvi_post_yearly - gric_ndvi_pre

print(gric_diff.dims)
print(gric_diff.coords)

In [13]:
# Plot the difference

diff_plot = (
    gric_diff.hvplot(x='x', y='y', cmap='PiYG', geo=True, groupby='year',
                     clim=(-0.5, 0.5),   
                     title = 'Gila River Indian Community NDVI\n'
                     'Comparing 2001-2004 to Following Years')
    *
    gric_gdf.hvplot(geo=True, fill_color=None, line_color='black')
)

diff_plot

In [14]:
years = list(diff_plot.kdims[0].values)
years.sort()     # just to be safe

diff_plot_bottom = pn.panel(
    diff_plot,
    widget_location='bottom',
    widgets={
        'year': pn.widgets.DiscreteSlider(
            name='year',
            options=years,
            value=years[0]
        )
    }
)

diff_plot_bottom

In [ ]:
diff_plot_bottom.save('img/gric_diff_plot_orginal.html', embed=True)

In [15]:
cumulative_sum = gric_ndvi_post_yearly.cumsum('year')

len(cumulative_sum)

In [16]:
year_count = xr.DataArray(
    range(1, len(gric_ndvi_post_yearly.year) + 1),
    coords=[gric_ndvi_post_yearly.year],
    dims=['year']
)

len(year_count)

In [17]:
gric_run_avg = cumulative_sum / year_count

len(gric_run_avg)

In [18]:
new_gric_diff = gric_run_avg - gric_ndvi_pre

print(new_gric_diff.dims)
print(new_gric_diff.coords)

In [ ]:
new_diff_plot = (
    new_gric_diff.hvplot(x='x', y='y', cmap='PiYG', geo=True, groupby='year',
                     clim=(-0.5, 0.5),   
                     title = 'Gila River Indian Community NDVI\n'
                     'Comparing 2001-2004 to Average of Following Years')
    *
    gric_gdf.hvplot(geo=True, fill_color=None, line_color='black')
)

years = list(diff_plot.kdims[0].values)
years.sort()

new_diff_plot_bottom = pn.panel(
    new_diff_plot,
    widget_location='bottom',
    widgets={
        'year': pn.widgets.DiscreteSlider(
            name='year',
            options=years,
            value=years[0]
        )
    }
)

new_diff_plot_bottom